In [1]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.models.gpt2.modeling_gpt2 import GPT2Model

g:\software\anaconda\envs\pytorch\Lib\site-packages\torch\cuda\__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
model_name = "G:/model_weights/models/model/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [7]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [3]:
# Define PAD Token = EOS Token = 50256
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

# pad on the left so we can append new tokens on the right
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

In [4]:
def get_float_dtype(self):
    return torch.float32

GPT2Model.dtype = property(get_float_dtype)

In [5]:
model.get_memory_footprint()

8044936704

In [41]:
def quantize(t):
    min_val, max_val = t.min(), t.max()
    scale = (max_val - min_val) / 255
    zero_point = min_val

    state = (scale, zero_point)
    t_quant = (t - zero_point) / scale
    t_quant = torch.clamp(t_quant, min=0, max=255)
    t_quant = t_quant.type(torch.uint8)
    return t_quant, state

In [21]:
t = model.model.layers[0].self_attn.q_proj.weight.data

In [23]:
t, t.shape

(tensor([[-0.0014, -0.0125,  0.0090,  ..., -0.0052, -0.0126,  0.0018],
         [ 0.0293, -0.0003, -0.0099,  ..., -0.0045,  0.0056,  0.0189],
         [-0.0183,  0.0026, -0.0280,  ..., -0.0057, -0.0027, -0.0095],
         ...,
         [ 0.0217, -0.0132, -0.0249,  ..., -0.0178,  0.0364,  0.0282],
         [-0.0281,  0.0131,  0.0140,  ..., -0.0260, -0.0027, -0.0466],
         [ 0.0090,  0.0508, -0.0120,  ...,  0.0354, -0.0210, -0.0186]],
        dtype=torch.bfloat16),
 torch.Size([4096, 2560]))

In [28]:
t_q, state = quantize(t)
t_q, state

(tensor([[145, 142, 147,  ..., 144, 142, 145],
         [152, 145, 142,  ..., 144, 146, 150],
         [141, 146, 139,  ..., 144, 144, 143],
         ...,
         [151, 142, 140,  ..., 141, 154, 152],
         [139, 148, 149,  ..., 139, 144, 134],
         [147, 158, 142,  ..., 154, 141, 141]], dtype=torch.uint8),
 (tensor(0.0041, dtype=torch.bfloat16), tensor(-0.5898, dtype=torch.bfloat16)))

In [ ]:
def dequantize(t, state):
    scale, zero_point = state
    return t.to(torch.bfloat16) * scale + zero_point

In [30]:
dequantize(t_q, state)

tensor([[-0.0013, -0.0135,  0.0068,  ..., -0.0054, -0.0135, -0.0013],
        [ 0.0271, -0.0013, -0.0135,  ..., -0.0054,  0.0027,  0.0190],
        [-0.0175,  0.0027, -0.0257,  ..., -0.0054, -0.0054, -0.0094],
        ...,
        [ 0.0230, -0.0135, -0.0216,  ..., -0.0175,  0.0352,  0.0271],
        [-0.0257,  0.0109,  0.0149,  ..., -0.0257, -0.0054, -0.0460],
        [ 0.0068,  0.0515, -0.0135,  ...,  0.0352, -0.0175, -0.0175]])

In [32]:
t - dequantize(t_q, state)

tensor([[-9.1553e-05,  1.0376e-03,  2.1667e-03,  ...,  1.5259e-04,
          9.1553e-04,  3.1433e-03],
        [ 2.1973e-03,  1.0452e-03,  3.6011e-03,  ...,  8.8501e-04,
          2.8381e-03, -6.1035e-05],
        [-7.6294e-04, -1.0681e-04, -2.2888e-03,  ..., -3.0518e-04,
          2.6398e-03, -9.1553e-05],
        ...,
        [-1.3123e-03,  3.0518e-04, -3.2959e-03,  ..., -2.7466e-04,
          1.1597e-03,  1.0986e-03],
        [-2.4109e-03,  2.1973e-03, -8.8501e-04,  ..., -3.3569e-04,
          2.7008e-03, -6.7139e-04],
        [ 2.1667e-03, -6.7139e-04,  1.5259e-03,  ...,  1.8311e-04,
         -3.4485e-03, -1.0071e-03]])

In [39]:
def quantize_model(model):
    states = {}
    for name, param in model.named_parameters():
        param.requires_grad = False # int8 类型的数据不可微分
        param.data, state = quantize(param.data)
        states[name] = state
    return model, states

In [35]:
def size_in_bytes(t):
    return t.numel() * t.element_size()

In [45]:
t, states = quantize_model(model)
sum([
    size_in_bytes(v[0] + size_in_bytes(v[1])) for v in states.values()
])

1592

In [43]:
model.get_memory_footprint()

4022468608

In [46]:
def de_quantize_model(model):
    for named, param in model.named_parameters():
        param.data = dequantize(param.data, states[named])

    return model

In [47]:
de_quantize_model(model)
model.get_memory_footprint()

16089872896